# Qwen3-1.7B SFT LoRA — POC Triage Médical CHSA

Fine-tuning supervisé du modèle **Qwen3-1.7B-Base** avec LoRA pour l'assistance au triage médical des urgences du Centre Hospitalier Saint-Aurélien.

**Pipeline :** Qwen3-1.7B-Base → SFT (LoRA) → DPO → Endpoint vLLM  
**Repo :** [XavierCoulon/OC_P14_Finetunez_votre_propre_LLM](https://github.com/XavierCoulon/OC_P14_Finetunez_votre_propre_LLM)  
**Modèle publié :** [XavierCoulon/qwen3-1.7b-chsa-sft-lora](https://huggingface.co/XavierCoulon/qwen3-1.7b-chsa-sft-lora)

> Exécuter sur **Kaggle** ou **Google Colab** avec GPU (T4 minimum).  
> Prérequis Secrets : `HF_TOKEN`, `WANDB_API_KEY`.

In [ ]:
%%capture
import os, re

env_keys = "".join(os.environ.keys())
ON_COLAB  = "COLAB_" in env_keys
ON_KAGGLE = "KAGGLE_" in env_keys

if ON_COLAB or ON_KAGGLE:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
else:
    !pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install wandb rouge_score

In [ ]:
def get_secret(name):
    """Lit un secret depuis Kaggle, Colab ou variable d'environnement."""
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    if ON_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    return os.environ.get(name, "")

import wandb
wandb.login(key=get_secret('WANDB_API_KEY'))
wandb.init(project="chsa-sft-qwen3", name=f"run-{__import__('datetime').datetime.now().strftime('%Y%m%d-%H%M')}")

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None         # None = auto (Float16 pour T4, Bfloat16 pour Ampere+)
load_in_4bit = True  # 4-bit quantization pour réduire la VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

<a name="Data"></a>
### Data Prep

Chargement du dataset médical bilingue depuis HuggingFace Hub (`XavierCoulon/oc-p14-dataset`, split `sft`).

**Thinking Mode Qwen3 :** Qwen3 supporte un mode raisonnement activé via `/think` (ou désactivé via `/no_think`) dans le prompt utilisateur. Pour préserver cette capacité après SFT, on mixe le dataset : ~75 % des exemples avec `/think`, ~25 % avec `/no_think`.

In [ ]:
import random
random.seed(42)

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    responses    = examples["response"]
    texts = []
    for instruction, response in zip(instructions, responses):
        # Thinking Mode : 75 % /think, 25 % /no_think (issue #8)
        think_tag = "/think" if random.random() < 0.75 else "/no_think"
        text = (
            f"<|im_start|>user\n{think_tag}\n{instruction}<|im_end|>\n"
            f"<|im_start|>assistant\n{response}<|im_end|>"
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

from datasets import load_dataset

dataset = load_dataset("XavierCoulon/oc-p14-dataset", "sft", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Train : {len(dataset)} exemples")

val_dataset = load_dataset("XavierCoulon/oc-p14-dataset", "sft", split="val")
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)
print(f"Val   : {len(val_dataset)} exemples")

print("\nExemple :")
print(dataset[0]["text"][:400])


<a name="Train"></a>
### Entraînement

SFTTrainer avec LoRA — 3 epochs sur les 4 324 paires SFT. Checkpoints tous les 200 steps (reprise possible). Métriques loggées sur W&B.

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,   # batch effectif = 8
        num_train_epochs = 3,              # 2→3 : avec 5033 samples, eval loss encore en descente à epoch 1.91 (run-20260527-1118)
                                           # (avait été réduit à 2 sur base du run-20260523-0701 avec 4324 samples, plateau à epoch 1.85)
        warmup_ratio = 0.1,
        learning_rate = 2e-4,
        logging_steps = 25,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs",
        report_to = "wandb",
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 3,
        eval_strategy = "steps",
        eval_steps = 200,
    ),
)


In [ ]:
trainer_stats = trainer.train()

In [21]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

5866.9184 seconds used for training.
97.78 minutes used for training.
Peak reserved memory = 14.191 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 97.446 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inférence

Test du modèle fine-tuné sur un cas de triage médical.

**Paramètres Qwen3 thinking :** `temperature=0.6`, `top_p=0.95`, `top_k=20` — ne pas utiliser le greedy decoding.

In [ ]:
FastLanguageModel.for_inference(model)

prompt = (
    "<|im_start|>user\n/think\n"
    "Patient : homme 52 ans, douleur thoracique irradiant dans le bras gauche depuis 30 min, "
    "sueurs froides, antécédent d'HTA.<|im_end|>\n"
    "<|im_start|>assistant\n"
)

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    temperature = 0.6,
    top_p = 0.95,
    top_k = 20,
    repetition_penalty = 1.2,
    use_cache = True,
)
print(tokenizer.batch_decode(outputs)[0])

<a name="Save"></a>
### Sauvegarde et push vers HuggingFace Hub

Sauvegarde locale des LoRA adapters puis push vers le Hub.

> Prérequis : `HF_TOKEN` dans les Colab Secrets.

In [ ]:
HF_TOKEN = get_secret('HF_TOKEN')
HF_REPO  = "XavierCoulon/qwen3-1.7b-chsa-sft-lora"

# Sauvegarde locale des adapters LoRA
model.save_pretrained("qwen_lora")
tokenizer.save_pretrained("qwen_lora")

# Push vers HuggingFace Hub
model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"Modèle publié : https://huggingface.co/{HF_REPO}")

### Chargement des adapters LoRA (optionnel)

Pour recharger les adapters sans ré-entraîner :

In [ ]:
if False:  # Mettre True pour charger sans ré-entraîner
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_lora",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)

In [ ]:
# Export merged 16bit pour DPO (référence correcte) et vLLM
model.push_to_hub_merged(
    f"{HF_REPO}-merged",
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN,
)
print(f"Modèle SFT merged publié : https://huggingface.co/{HF_REPO}-merged")

# Export GGUF pour llama.cpp / Ollama (décommenter selon le format souhaité)
# model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method="q4_k_m")
# model.push_to_hub_gguf(f"{HF_REPO}_gguf", tokenizer, quantization_method="q4_k_m", token=HF_TOKEN)


In [ ]:
wandb.finish()

In [ ]:
from datasets import load_dataset as load_eval_dataset
from rouge_score import rouge_scorer
import pandas as pd

eval_dataset = load_eval_dataset("XavierCoulon/oc-p14-dataset", "eval_clinique", split="eval")

FastLanguageModel.for_inference(model)
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
scores, results = [], []

for example in eval_dataset:
    prompt = (
        f"<|im_start|>user\n/no_think\n{example['instruction']}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    # Tronquer à max_seq_length - 256 pour laisser de la place à la génération
    if inputs["input_ids"].shape[1] > max_seq_length - 256:
        inputs = {k: v[:, -(max_seq_length - 256):] for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_new_tokens=256,
                             temperature=0.6, top_p=0.95, top_k=20,
                             repetition_penalty=1.2, use_cache=True)
    generated = tokenizer.batch_decode(outputs)[0]
    response = generated.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    score = scorer.score(example['response'], response)['rougeL'].fmeasure
    scores.append(score)
    results.append({
        "instruction": example['instruction'][:80] + "...",
        "reference":   example['response'][:120] + "...",
        "generated":   response[:120] + "...",
        "rougeL":      round(score, 3),
    })

mean_rougeL = sum(scores) / len(scores)
print(f"ROUGE-L moyen ({len(eval_dataset)} cas) : {mean_rougeL:.3f}")
pd.set_option('display.max_colwidth', 80)
print("\n--- 5 premiers exemples ---")
print(pd.DataFrame(results[:5]).to_string(index=False))

<a name="Evaluation"></a>
### Évaluation — eval_clinique (100 cas)

Score ROUGE-L du modèle fine-tuné sur le jeu d'évaluation clinique séparé (aucun overlap avec les données d'entraînement).